In [34]:
#Импорты библиотек
import pandas as pd
pd.set_option('display.max_colwidth', 100)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.pipeline import make_pipeline

from pathlib import Path

import joblib


In [35]:
#Чтение синтетических данных
df_train = pd.read_csv(Path("data") / "train.csv")
df_test = pd.read_csv(Path("data") / "test.csv")


In [36]:
#Модель
model = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5)),
    LogisticRegression(max_iter=1000),
)
#Обучение
model.fit(df_train["text"], df_train["label"])

labeled_test = df_test.dropna(subset=["label"]).copy()
labeled_test["prediction"] = model.predict(labeled_test["text"])

y_true = labeled_test["label"]
y_pred = labeled_test["prediction"]

print(f"Macro-F1 на {len(labeled_test)} размеченных отзывах: "
      f"{f1_score(y_true, y_pred, average='macro'):.3f}")
print(classification_report(y_true, y_pred, zero_division=0))

errors = labeled_test.loc[
    y_true.ne(y_pred),
    ["feedback_id", "text", "label", "prediction"],
]

Macro-F1 на 24 размеченных отзывах: 0.841
                       precision    recall  f1-score   support

    График и нагрузка       1.00      1.00      1.00         4
Доступы и инструменты       1.00      0.75      0.86         4
    Задачи и ожидания       0.75      0.75      0.75         4
  Обучение и практика       1.00      0.75      0.86         4
 Поддержка наставника       0.57      1.00      0.73         4
   Трудностей не было       1.00      0.75      0.86         4

             accuracy                           0.83        24
            macro avg       0.89      0.83      0.84        24
         weighted avg       0.89      0.83      0.84        24



In [ ]:
#ошибки модели
errors


,feedback_id,text,label,prediction
0,TEST-001,Мой профиль так и не появился в списке пользователей весов. Все коробки за меня взвешивал сотруд...,Доступы и инструменты,Поддержка наставника
13,TEST-014,"Мне объяснили, где смотреть историю клиента, но ни разу не дали провести пробный звонок. На перв...",Обучение и практика,Задачи и ожидания
15,TEST-016,"Я думал, что после сканирования заказ надо отнести на упаковку, а коллега ждал, что я сначала от...",Задачи и ожидания,Поддержка наставника
17,TEST-018,"В зале первое время спрашивал по мелочи. На вопросы отвечали сразу, ничего из-за этого не задерж...",Трудностей не было,Поддержка наставника


In [ ]:
#сохранение модели
project_dir = Path.cwd()

model_path = project_dir / "model.joblib"
joblib.dump(model, model_path)
print(f"Модель сохранена в {model_path}")

In [39]:
#предскаания на демо версии обученой модели
demo = pd.read_csv(project_dir / "data" / "demo.csv")
demo["prediction"] = model.predict(demo["text"])

store_demo = demo.loc[
    demo["department"].eq("Магазины"),
    ["feedback_id", "text", "prediction"],
]


display(store_demo)

,feedback_id,text,prediction
8,DEMO-009,"Первый возврат оформлял при покупателе. До этого я видел только, как пробивают обычную продажу.",Обучение и практика
9,DEMO-010,"Правила возврата нам зачитали, а пройти операцию на учебной кассе я так и не попробовал.",Обучение и практика
10,DEMO-011,"На обучении пропустили возврат товара, купленного онлайн. При первом таком клиенте я растерялся.",Обучение и практика
11,DEMO-012,"Инструкцию по примерке экипировки прочитал, но полезно было бы разок потренироваться на примере ...",Обучение и практика
12,DEMO-013,Вкладка с остатками на моём планшете не открывалась из-за прав. Приходилось спрашивать у кассира.,Доступы и инструменты
13,DEMO-014,"На вечер выдали задачи по залу и поставке, без порядка. Какую бросать, если клиентов много, я не...",Задачи и ожидания
14,DEMO-015,"Пока всё хорошо, по ассортименту ориентируюсь и вопросы решаю по ходу.",Трудностей не было
15,DEMO-016,"В субботу после открытия добавили ещё четыре часа работы, я об этом заранее не знал.",График и нагрузка
